In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from model import Transformer

from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

import torch.nn.functional as F

In [2]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/Sampuran01/english-japanese/" + splits["train"])
df.head()

,translation
0,"{'en': 'Yeah, Vincent Hanna.', 'ja': '- ラウール -..."
1,{'en': 'I'm being held in a basement. I've bee...
2,"{'en': 'It works!', 'ja': '動いたよ！'}"
3,{'en': 'I'm just trying to find out what happe...
4,"{'en': 'You okay?', 'ja': '無事か？'}"


In [3]:
df['translation'].iloc[1, ]['ja']

'いま地下に居ます 他の2人と一緒に誘拐されたんです！'

In [4]:
def get_all_sentences(df, lang):
    for item in df[lang]:
        yield item

def get_or_build_tokenizer(df, lang):
    # config['tokenizer_file'] = '../tokenizers/tokenizer_{0}.json'
    tokenizer_path = Path(("tokenizer_{0}.json".format(lang)))
    if not Path.exists(tokenizer_path):
        tokenizer = Tokenizer(WordLevel(unk_token='[UNK]'))
        tokenizer.pre_tokenizer = Whitespace()
        trainer = WordLevelTrainer(special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]"], min_frequency=2)
        tokenizer.train_from_iterator(get_all_sentences(df, lang), trainer=trainer)
        tokenizer.save(str(tokenizer_path))
    else:
        tokenizer = Tokenizer.from_file(str(tokenizer_path))
    
    return tokenizer

In [5]:
tokenizer_src = get_or_build_tokenizer(df, 'en')
tokenizer_tgt = get_or_build_tokenizer(df, 'jp')

In [6]:
def causal_mask(size):
    mask = torch.triu(torch.ones(1, size, size), diagonal=1).type(torch.int)
    return mask == 0


class JpEnDataset(Dataset):
    def __init__(self, df, tokenizer_src, tokenizer_tgt, seq_len):
        super().__init__()
        self.df = df
        self.tokenizer_src = tokenizer_src
        self.tokenizer_tgt = tokenizer_tgt
        self.seq_len = seq_len
        self.sos_token = torch.tensor([tokenizer_src.token_to_id('[SOS]')], dtype=torch.int64)
        self.eos_token = torch.tensor([tokenizer_src.token_to_id('[EOS]')], dtype=torch.int64)
        self.pad_token = torch.tensor([tokenizer_src.token_to_id('[PAD]')], dtype=torch.int64)
    
    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        src_text = self.df['translation'].iloc[index, ]['en']
        tgt_text = self.df['translation'].iloc[index, ]['ja']

        enc_input_tokens = self.tokenizer_src.encode(src_text).ids
        dec_input_tokens = self.tokenizer_tgt.encode(tgt_text).ids

        enc_num_padding_tokens = self.seq_len - len(enc_input_tokens) - 2
        dec_num_padding_tokens = self.seq_len - len(dec_input_tokens) - 1

        if enc_num_padding_tokens < 0 or dec_num_padding_tokens < 0:
            raise ValueError("sentence is too long")

        # add sos and eos to the source text
        encoder_input = torch.cat([
            self.sos_token,
            torch.tensor(enc_input_tokens, dtype=torch.int64),
            self.eos_token,
            torch.tensor([self.pad_token] * enc_num_padding_tokens, dtype=torch.int64)
        ])

        # add sos to decoder input
        decoder_input = torch.cat([
            self.sos_token,
            torch.tensor(dec_input_tokens, dtype=torch.int64),
            torch.tensor([self.pad_token] * dec_num_padding_tokens, dtype=torch.int64)
        ])

        # add eos to decoder output
        label = torch.cat([
            torch.tensor(dec_input_tokens, dtype=torch.int64),
            self.eos_token,
            torch.tensor([self.pad_token] * dec_num_padding_tokens, dtype=torch.int64)
        ])

        assert encoder_input.size(0) == self.seq_len
        assert decoder_input.size(0) == self.seq_len
        assert label.size(0) == self.seq_len

        return {
            "encoder_input": encoder_input,
            "decoder_input": decoder_input,
            "encoder_mask": (encoder_input != self.pad_token).unsqueeze(0).unsqueeze(0).int(), # (1, 1, seq_len)
            "decoder_mask": (decoder_input != self.pad_token).unsqueeze(0).unsqueeze(0).int() & causal_mask(decoder_input.size(0)), # (1, seq_len) & (1, seq_len, seq_len)
            "label": label,
            "src_text": src_text,
            "tgt_text": tgt_text
        }

In [12]:
# Keep 90% for training and 10% for validation
train_df, val_df = train_test_split(df, test_size=0.8, random_state=42)

train_ds = JpEnDataset(train_df, tokenizer_src, tokenizer_tgt, 256)
val_ds = JpEnDataset(val_df, tokenizer_src, tokenizer_tgt, 256)

train_loader= DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader= DataLoader(val_ds, batch_size=1, shuffle=True)

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
model = Transformer(src_vocab_size=tokenizer_src.get_vocab_size(), tgt_vocab_size=tokenizer_tgt.get_vocab_size()).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, eps=1e-9)
loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer_src.token_to_id('[PAD]'), label_smoothing=0.1).to(device)

In [14]:
device = torch.device('cuda')
model = model.to(device)

for epoch in range(5):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        encoder_input = batch['encoder_input'].to(device) # (B, seq_len)
        decoder_input = batch['decoder_input'].to(device) # (B, seq_len)
        encoder_mask = batch['encoder_mask'].to(device)   # (B, 1, 1, seq_len)
        decoder_mask = batch['decoder_mask'].to(device)   # (B, 1, seq_len, seq_len)
        label = batch['label'].to(device)   # (B, seq_len)

        output = model(encoder_input, decoder_input, encoder_mask, decoder_mask)
        loss = loss_fn(output.view(-1, tokenizer_tgt.get_vocab_size()), label.view(-1))
        loss.backward()

        # Gradient clipping (optional)
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

        if (batch_idx + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{10}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}\r', end="")

    avg_loss = total_loss / len(train_loader)

    torch.save({
        "epoch": epoch,
        'model_state_dict': model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, f"weights\\en-jp_tmodel_{epoch}.pth")
    
    print(f'Epoch [{epoch+1}/{10}], Average Loss: {avg_loss:.4f}')

ValueError: sentence is too long

In [34]:
def translate(model, src_text, tokenizer_src, tokenizer_tgt, max_len=100):
    model.eval()

    with torch.no_grad():
        enc_input_tokens = tokenizer_src.encode(src_text).ids
        enc_num_padding_tokens = max_len - len(enc_input_tokens) - 2
        pad_token = torch.tensor([tokenizer_src.token_to_id('[PAD]')]).to(device)

        encoder_input = torch.cat([
                torch.tensor([tokenizer_src.token_to_id('[SOS]')]),
                torch.tensor(enc_input_tokens),
                torch.tensor([tokenizer_src.token_to_id('[EOS]')]),
                torch.tensor([pad_token] * enc_num_padding_tokens)
            ]).unsqueeze(0).to(device)
        
        src_mask = (encoder_input != pad_token).unsqueeze(0).unsqueeze(0).to(device)

        decoder_input = torch.empty(1, 1).fill_(tokenizer_src.token_to_id('[SOS]')).to(torch.int64).to(device)

        for i in range(max_len):
            if decoder_input.size(1) == max_len: break

            decoder_mask = causal_mask(decoder_input.size(1)).type_as(src_mask).to(device)

            output = model(encoder_input, decoder_input, src_mask, decoder_mask)

            next_token_logits = output[:, -1, :]
            print(next_token_logits)

            next_word = torch.argmax(F.softmax(next_token_logits, dim=-1), dim=-1)
            print(next_word)

            decoder_input = torch.cat([decoder_input, torch.empty(1, 1).type_as(decoder_input).fill_(next_word.item()).to(device)], dim=1)

            if next_word.item() == tokenizer_src.token_to_id('[EOS]'):
                break
    

    return decoder_input


In [ ]:
src_text = "島"
print(tokenizer_src.encode(src_text).ids)
torch.empty(1, 1).fill_(tokenizer_src.token_to_id('[SOS]'))

In [ ]:
w = translate(model, src_text, tokenizer_src, tokenizer_tgt)

In [ ]:
w

In [ ]:
for c in w[0]:
    print(tokenizer_tgt.id_to_token(c))

In [48]:
torch.save({
        "epoch": 5,
        'model_state_dict': model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, "weights\\t_model_5.pth")